In [0]:
%pip install xgboost seaborn --quiet
dbutils.library.restartPython()

In [0]:
# =============================================================================
# ICEBASE — PHASE 4 | NOTEBOOK 03
# ML Model 2 — XGBoost Churn Prediction
# Idaho Mashers Hockey Analytics Platform
# =============================================================================
# STANDALONE NOTEBOOK — Run on a DEDICATED (single-user) cluster.
#
# WHAT THIS NOTEBOOK DOES:
#   Trains an XGBoost binary classifier to predict fan churn.
#   Target variable: churn_flag from icebase.gold.fan_features
#   (1 = churned: no purchase in 45+ days, 0 = retained)
#
#   Uses the Feature Engineering client so MLflow can track feature lineage
#   and the model can be scored against the feature table at inference time.
#
# MODEL DETAILS:
#   Algorithm:    XGBoost (xgboost.XGBClassifier)
#   Features:     RFM + behavioral signals from icebase.gold.fan_features
#   Target:       churn_flag (binary: 1=churned, 0=active)
#   Tracking:     MLflow — parameters, metrics, confusion matrix logged
#   Registry:     icebase.gold.xgboost_churn_predictor
#
# RISK TIERS (assigned at scoring time):
#   High Risk   : churn_probability >= 0.70
#   Medium Risk : churn_probability >= 0.40 and < 0.70
#   Low Risk    : churn_probability < 0.40
#
# OUTPUT TABLE:
#   icebase.gold.ml_churn_scores
#   Columns: customer_id, churn_probability, risk_tier, scored_at, model_version
# =============================================================================

In [0]:
# COMMAND ----------
# ── CELL 1: Imports ────────────────────────────────────────────────────────
 
import mlflow
import mlflow.xgboost
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, accuracy_score,
    classification_report, confusion_matrix
)
from sklearn.preprocessing import StandardScaler
from mlflow.models import infer_signature
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
 
from pyspark.sql import functions as F
 
mlflow.set_registry_uri("databricks-uc")
 
CATALOG          = "icebase"
GOLD             = f"{CATALOG}.gold"
FEATURE_TABLE    = f"{GOLD}.fan_features"
CHURN_TABLE      = f"{GOLD}.ml_churn_scores"
REGISTERED_MODEL = f"{GOLD}.xgboost_churn_predictor"
 
username = spark.sql("SELECT current_user()").collect()[0][0]
mlflow.set_experiment(f"/Users/{username}/icebase/phase4_churn")
 
print("✅ Churn prediction notebook loaded")
print(f"   Feature table:    {FEATURE_TABLE}")
print(f"   Output table:     {CHURN_TABLE}")
print(f"   Registered model: {REGISTERED_MODEL}")

In [0]:
# COMMAND ----------
# ── CELL 2: Build Training Dataset ──────────────────────────────────────────────
#
# Load features directly from the feature table.
# Select the features we need for churn prediction plus the label.
 
# Define which features to use for churn prediction
# NOTE: recency_days and cohort_days_since_last are excluded because they're
#       directly derived from the churn definition (45+ days since last purchase)
#       and would cause data leakage.
CHURN_FEATURES = [
    "frequency_games",
    "monetary_net",
    "promo_sensitivity",
    "promo_rate",
    "advance_purchase_rate",
    "avg_days_before_game",
    "avg_seat_tier_rank",
    "tenure_days",
    "is_season_holder",
    "channels_used",
    "jersey_night_attendee",
    "hot_start_buyer",
    "slump_buyer",
    "was_lapsed_reactivation",
    "returned_30d",
    "is_one_and_done",
    "is_jersey_night_cohort",
]
 
# Load feature table and convert to pandas
training_df = (
    spark.table(FEATURE_TABLE)
    .select(["customer_id", "churn_flag"] + CHURN_FEATURES)
    .toPandas()
)
 
print(f"✅ Training dataset built: {training_df.shape[0]:,} rows × {training_df.shape[1]} cols")
print(f"   Churn rate in training data: {training_df['churn_flag'].mean():.1%}")

In [0]:
# COMMAND ----------
# ── CELL 3: Train/Test Split ───────────────────────────────────────────────
 
X = training_df[CHURN_FEATURES].fillna(0)
y = training_df["churn_flag"]
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.20,
    random_state = 42,
    stratify     = y,     # Keep churn ratio consistent across splits
)
 
print(f"✅ Train/test split complete")
print(f"   Train: {len(X_train):,} rows | Test: {len(X_test):,} rows")
print(f"   Train churn rate: {y_train.mean():.1%}")
print(f"   Test churn rate:  {y_test.mean():.1%}")

In [0]:
# COMMAND ----------
# ── CELL 4: Train XGBoost Model ────────────────────────────────────────────────────────
#
# XGBoost hyperparameters are tuned for the IceBase data characteristics:
#   - scale_pos_weight handles class imbalance (more active fans than churned)
#   - max_depth=4 prevents overfitting on a 5k dataset
#   - use_label_encoder=False suppresses deprecation warning
#
# All parameters and metrics are logged to MLflow automatically.
# The model is registered to Unity Catalog under a 3-part name.
 
# Calculate scale_pos_weight to handle class imbalance
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = round(neg_count / pos_count, 2)
 
print(f"Class balance — Active: {neg_count:,} | Churned: {pos_count:,}")
print(f"scale_pos_weight: {scale_pos_weight}")
 
PARAMS = {
    "n_estimators":      300,
    "max_depth":         4,
    "learning_rate":     0.05,
    "subsample":         0.8,
    "colsample_bytree":  0.8,
    "scale_pos_weight":  scale_pos_weight,
    "random_state":      42,
    "eval_metric":       "auc",
    "early_stopping_rounds": 20,
}
 
with mlflow.start_run(run_name="xgboost_churn_v1") as run:
 
    # ── Enable MLflow autologging for XGBoost ────────────────────────────────────────
    mlflow.xgboost.autolog(log_models=False)  # We log manually below
 
    # ── Train ────────────────────────────────────────────────────────────────────────────────────
    model = xgb.XGBClassifier(**PARAMS)
    model.fit(
        X_train, y_train,
        eval_set       = [(X_test, y_test)],
        verbose        = 50,
    )
 
    # ── Evaluate ────────────────────────────────────────────────────────────────────────────────────
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred       = (y_pred_proba >= 0.5).astype(int)
 
    auc      = roc_auc_score(y_test, y_pred_proba)
    accuracy = accuracy_score(y_test, y_pred)
 
    # Log metrics
    mlflow.log_metrics({
        "test_auc":      round(auc, 4),
        "test_accuracy": round(accuracy, 4),
        "train_size":    len(X_train),
        "test_size":     len(X_test),
    })
 
    print(f"\n── Model Evaluation ──")
    print(f"  AUC:      {auc:.4f}  (target: >0.75)")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"\n{classification_report(y_test, y_pred, target_names=['Active','Churned'])}")
 
    # ── Confusion matrix as artifact ────────────────────────────────────────────────────────────────────
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["Active","Churned"],
        yticklabels=["Active","Churned"], ax=ax
    )
    ax.set_title("Churn Model — Confusion Matrix")
    ax.set_ylabel("Actual"); ax.set_xlabel("Predicted")
    plt.tight_layout()
    plt.savefig("/tmp/confusion_matrix.png", dpi=100)
    mlflow.log_artifact("/tmp/confusion_matrix.png")
    plt.close()
 
    # ── Feature importance as artifact ───────────────────────────────────────────────────────────────────────
    importance_df = pd.DataFrame({
        "feature":   CHURN_FEATURES,
        "importance": model.feature_importances_,
    }).sort_values("importance", ascending=False)
 
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(importance_df["feature"], importance_df["importance"], color="steelblue")
    ax.set_title("XGBoost Churn Model — Feature Importance")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig("/tmp/feature_importance.png", dpi=100)
    mlflow.log_artifact("/tmp/feature_importance.png")
    plt.close()
 
    print(f"\n── Top 5 churn predictors ──")
    print(importance_df.head(5).to_string(index=False))
 
    # ── Log model ───────────────────────────────────────────────────────────────────────────────────────────────
    input_example = X_test.iloc[:5]
    signature     = infer_signature(input_example, model.predict_proba(input_example))
 
    mlflow.xgboost.log_model(
        xgb_model             = model,
        artifact_path         = "churn_model",
        signature             = signature,
        input_example         = input_example,
        registered_model_name = REGISTERED_MODEL,
    )
 
    run_id = run.info.run_id
    print(f"\n✅ Model logged and registered")
    print(f"   Run ID:  {run_id}")
    print(f"   AUC:     {auc:.4f}")
    print(f"   Model:   {REGISTERED_MODEL}")

In [0]:
# COMMAND ----------
# ── CELL 5: Set Champion Alias in Model Registry ───────────────────────────
#
# In Unity Catalog, model aliases replace the old Staging/Production stages.
# Setting the "champion" alias marks this version as the production model.
# Downstream scoring notebooks reference "champion" — not a version number —
# so promoting a new model is just updating the alias, not changing code.
 
from mlflow import MlflowClient
client = MlflowClient()
 
# Get the latest version using search_model_versions (UC-compatible approach)
versions = client.search_model_versions(f"name='{REGISTERED_MODEL}'")
latest_version = max([int(v.version) for v in versions])
 
client.set_registered_model_alias(
    name    = REGISTERED_MODEL,
    alias   = "champion",
    version = str(latest_version),
)
 
print(f"✅ Alias 'champion' set on version {latest_version} of {REGISTERED_MODEL}")
print(f"   Load champion model with: mlflow.xgboost.load_model('models:/{REGISTERED_MODEL}@champion')")

In [0]:
# COMMAND ----------
# ── CELL 6: Score All Fans and Write ml_churn_scores ──────────────────────
#
# Load the registered champion model and score every fan in the feature table.
# Assign risk tier based on probability threshold.
# Write results to icebase.gold.ml_churn_scores.
 
print("Scoring all fans...")
 
# Load all features for scoring
all_features = spark.table(FEATURE_TABLE).toPandas()
X_all        = all_features[CHURN_FEATURES].fillna(0)
 
# Load champion model
champion_model = mlflow.xgboost.load_model(
    f"models:/{REGISTERED_MODEL}@champion"
)
churn_probs = champion_model.predict_proba(X_all)[:, 1]
 
# Build scoring DataFrame
scores_df = pd.DataFrame({
    "customer_id":       all_features["customer_id"].values,
    "churn_probability": churn_probs.round(4),
    "risk_tier": pd.cut(
        churn_probs,
        bins   = [-0.001, 0.40, 0.70, 1.001],
        labels = ["Low", "Medium", "High"]
    ).astype(str),
    "scored_at":         datetime.now(),
    "model_version":     f"champion_v{latest_version}",
    "model_name":        REGISTERED_MODEL,
})
 
# Write to Gold
(
    spark.createDataFrame(scores_df)
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(CHURN_TABLE)
)
 
total    = len(scores_df)
high     = (scores_df["risk_tier"] == "High").sum()
medium   = (scores_df["risk_tier"] == "Medium").sum()
low      = (scores_df["risk_tier"] == "Low").sum()
 
print(f"✅ ml_churn_scores written | {total:,} fans scored")
print(f"   High risk:   {high:,}  ({high/total:.1%})")
print(f"   Medium risk: {medium:,}  ({medium/total:.1%})")
print(f"   Low risk:    {low:,}  ({low/total:.1%})")

In [0]:
# COMMAND ----------
# ── CELL 7: Business Validation ───────────────────────────────────────────
#
# Join churn scores to segments to validate the story:
# Lapsed and Promo Hunter segments should have the highest churn risk.
# Season Core should have the lowest.
 
print("── Churn risk by fan segment ──")
display(
    spark.table(CHURN_TABLE)
    .join(spark.table(f"{GOLD}.ml_fan_segments"), "customer_id", "left")
    .groupBy("segment_label")
    .agg(
        F.count("*")                                        .alias("fans"),
        F.round(F.avg("churn_probability"), 3)              .alias("avg_churn_prob"),
        F.sum(F.when(F.col("risk_tier") == "High", 1).otherwise(0))
                                                            .alias("high_risk_count"),
        F.round(
            F.sum(F.when(F.col("risk_tier") == "High", 1).otherwise(0))
            / F.count("*") * 100, 1
        )                                                   .alias("high_risk_pct"),
    )
    .orderBy(F.col("avg_churn_prob").desc())
)
 
# EXPECTED:
#   Lapsed        → highest avg_churn_prob, highest high_risk_pct
#   Promo Hunter  → second highest avg_churn_prob
#   Season Core   → lowest avg_churn_prob
#   This validates the model captures the narrative baked into the data